In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import metpy.calc as mpcalc
import numpy as np
import xarray as xr
import glob
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import pandas as pd
import cmocean.cm as cmo

In [2]:
# get dataset 


y =np.load("/work/uo1075/u241321/data/y310_T.npy") 

data_x = xr.open_dataset('/work/uo1075/u241321/data/uas_1969-2019_assi_dt.nc', decode_times=False)  # unit: m/s
data_y = xr.open_dataset('/work/uo1075/u241321/data/vas_1969-2019_assi_dt.nc', decode_times=False)

var_x = np.mean(data_x['__xarray_dataarray_variable__'], axis=1) 
var_y = np.mean(data_y['__xarray_dataarray_variable__'], axis=1) 


In [7]:
# regression onto x and y transport respectively, then calculate manitude with direction

field_x = var_x.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space

nn = 6 # number of regression year


from sklearn.linear_model import LinearRegression
def regression(x,y):

    coef = LinearRegression(fit_intercept=True).fit(x.reshape(-1, 1), y.values.reshape(-1, 1)).coef_
    

    return coef

# regression, center on 4-47, 44 year (start from 0)

coe_x = np.zeros((nn, field_x.shape[1]))

for m in range(0,field_x.shape[1],1):
        coe_x[0,m] = regression(y[7:47], field_x[2:42,m])
        coe_x[1,m] = regression(y[7:47], field_x[3:43,m])
        coe_x[2,m] = regression(y[7:47], field_x[4:44,m])
        coe_x[3,m] = regression(y[7:47], field_x[5:45,m])
        coe_x[4,m] = regression(y[7:47], field_x[6:46,m])
        coe_x[5,m] = regression(y[7:47], field_x[7:47,m])
       


In [8]:
coe_x = xr.DataArray(coe_x,  
                    dims=['mode','spatial'],
                    coords=dict(
                        spatial=field_x.spatial,
                         mode=np.arange(1,nn+1,1))
                    , )
# field = var.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space
spatial = field_x .coords["spatial"]
mode = coe_x .coords["mode"]
reg_x = xr.DataArray(coe_x, dims = ["mode","spatial"], coords = {"mode":mode,"spatial":spatial}).unstack()  


In [9]:
field_y = var_y.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space
coe_y = np.zeros((nn, field_y.shape[1]))

for m in range(0,field_y.shape[1],1):
        coe_y[0,m] = regression(y[7:47], field_y[2:42,m])
        coe_y[1,m] = regression(y[7:47], field_y[3:43,m])
        coe_y[2,m] = regression(y[7:47], field_y[4:44,m])
        coe_y[3,m] = regression(y[7:47], field_y[5:45,m])
        coe_y[4,m] = regression(y[7:47], field_y[6:46,m])
        coe_y[5,m] = regression(y[7:47], field_y[7:47,m])


In [10]:
coe_y = xr.DataArray(coe_y,  
                    dims=['mode','spatial'],
                    coords=dict(
                        spatial=field_y.spatial,
                         mode=np.arange(1,nn+1,1))
                    , )

spatial_y = field_y .coords["spatial"]
reg_y = xr.DataArray(coe_y, dims = ["mode","spatial"], coords = {"mode":mode,"spatial":spatial_y}).unstack()        
lon = reg_x.lon
lat = reg_x.lat

In [11]:
reg_x.to_netcdf("/work/uo1075/u241321/data/reg_windx_T_c2_bp.nc")
reg_y.to_netcdf("/work/uo1075/u241321/data/reg_windy_T_c2_bp.nc")